# Streaming a LangGraph run

This notebook builds a small two-step graph (a poem pipeline: topic ->
outline -> poem) and then calls `graph.stream(...)` on it with different
`stream_mode` values to see what each one reports while the graph is running:
`"values"` (full state after each step), `"updates"` (only what each node
changed), `"messages"` (token-by-token LLM output), `"custom"` (progress events a
node emits itself), and multiple modes requested together.

Imports: the usual graph pieces, plus `get_stream_writer` for emitting custom progress events from inside a node.

In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langgraph.config import get_stream_writer
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

In [ ]:
load_dotenv()

In [ ]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0.7)

## The graph: outline -> write_poem

State holds a topic, a short deterministic outline, and the final poem. `outline_step` needs no LLM call — it just builds a short instruction string — so only `write_poem` costs an LLM call per run. Both nodes emit a custom progress event with `get_stream_writer()`, which has nothing to do with the graph state — it is a side-channel message meant only for whoever is watching the stream.

In [ ]:
class PoemState(TypedDict):
    topic: str
    outline: str
    poem: str

In [ ]:
def outline_step(state: PoemState) -> dict:
    writer = get_stream_writer()
    writer({"progress": f"outlining a poem about {state['topic']}..."})
    return {"outline": f"A cheerful 3-line poem about {state['topic']}."}


def write_poem(state: PoemState) -> dict:
    writer = get_stream_writer()
    writer({"progress": "writing the poem..."})

    response = llm.invoke(state["outline"])
    return {"poem": response.content}

In [ ]:
poem_graph = StateGraph(PoemState)
poem_graph.add_node("outline_step", outline_step)
poem_graph.add_node("write_poem", write_poem)
poem_graph.add_edge(START, "outline_step")
poem_graph.add_edge("outline_step", "write_poem")
poem_graph.add_edge("write_poem", END)

poem_app = poem_graph.compile()
poem_app

## `stream_mode="values"`

Yields the full state after every step. Each item is the complete state dict at that point in the run.

In [ ]:
for chunk in poem_app.stream({"topic": "cats", "outline": "", "poem": ""}, stream_mode="values"):
    print(chunk)

## `stream_mode="updates"`

Yields only what changed at each step, keyed by node name. This is usually easier to read than `"values"` because it does not repeat the whole state every time.

In [ ]:
for chunk in poem_app.stream({"topic": "robots", "outline": "", "poem": ""}, stream_mode="updates"):
    print(chunk)

## `stream_mode="messages"`

Streams LLM output token-by-token as it is generated, instead of waiting for the node to finish. Each item is a tuple `(message_chunk, metadata)`. `metadata["langgraph_node"]` says which node the chunk came from. Here only `write_poem` calls an LLM, so filtering on it just picks out the one real source of tokens — in a graph with more than one LLM node, this filter is how you pick which node's output to show. The Streamlit app in `11_Langgraph_Chatbot/frontend.py` already uses this mode to stream a chatbot's reply to the page as it is written.

In [ ]:
for message_chunk, metadata in poem_app.stream(
    {"topic": "dogs", "outline": "", "poem": ""}, stream_mode="messages"
):
    if metadata["langgraph_node"] == "write_poem" and message_chunk.content:
        print(message_chunk.content, end="", flush=True)
print()

## `stream_mode="custom"`

Streams whatever a node passes to `get_stream_writer()`. This is separate from the graph state — it is a way for a node to report progress ("outlining...", "writing...", etc.) to whoever is consuming the stream, without putting that message into the state itself. No LLM call is needed to see this mode, since both nodes write a progress event regardless of whether they call the LLM.

In [ ]:
for chunk in poem_app.stream({"topic": "the ocean", "outline": "", "poem": ""}, stream_mode="custom"):
    print(chunk)

## Multiple modes at once

Passing a list to `stream_mode` yields both kinds of event, and each item becomes a `(mode, chunk)` tuple so you can tell which stream it came from.

In [ ]:
for mode, chunk in poem_app.stream(
    {"topic": "pizza", "outline": "", "poem": ""}, stream_mode=["updates", "custom"]
):
    print(mode, "->", chunk)